# DePatch Celeb-FBI

Run `../setup_venv.sh` and `../download_dataset.sh` before using this notebook. Select the `Python (portable-depatch)` kernel.

In [ ]:
from pathlib import Path
import contextlib
import csv
import io
import os
import sys

import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from tqdm.notebook import tqdm as notebook_tqdm

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import depatch
depatch.tqdm = notebook_tqdm
from depatch import DePatchTrainer, PatchTrainerConfig


In [ ]:
dataset_dir = Path("datasets/celeb_fbi_640/images")
images = sorted(dataset_dir.glob("*.jpg"))
if not images:
    raise FileNotFoundError("Run ./download_dataset.sh first")


In [ ]:
config = PatchTrainerConfig(
    train_dir='datasets/celeb_fbi_640/images',
    val_dir='datasets/celeb_fbi_640/images',
    weights='yolo11s.pt',
    device='auto',
    output_dir='outputs/depatch_celeb_fbi',
    iterations=None,
    epochs=2000,
    batch_size=64,
    num_workers=4,
    use_train_as_val=True,
    train_eval_samples=256,
    val_eval_samples=256,
    eval_interval=100,
    cleanup_interval=0,
    cleanup_batch_interval=100,
    log_interval=50,
)

# Change batch size here if needed, e.g. config.batch_size = 32

In [ ]:
history = []
plot_handle = None


def read_history_csv(path=None):
    path = Path(config.output_dir) / "history.csv" if path is None else Path(path)
    if not path.exists():
        return []
    rows = []
    with path.open(newline="") as handle:
        for row in csv.DictReader(handle):
            parsed = {}
            for key, value in row.items():
                try:
                    number = float(value)
                    parsed[key] = int(number) if number.is_integer() else number
                except (TypeError, ValueError):
                    parsed[key] = value
            rows.append(parsed)
    return rows


def render_progress(rows, save=True):
    global plot_handle
    if not rows:
        return
    steps = [row.get("iteration", row["epoch"]) for row in rows]
    x_label = "iteration" if any("iteration" in row for row in rows) else "epoch"

    fig, (ax_loss, ax_asr) = plt.subplots(1, 2, figsize=(14, 4))
    ax_loss.plot(steps, [row["train_loss"] for row in rows], marker="o", label="train loss")
    ax_loss.set_title("Loss")
    ax_loss.set_xlabel(x_label)
    ax_loss.set_ylabel("loss")
    ax_loss.grid(True, alpha=0.25)
    ax_loss.legend()

    ax_asr.plot(steps, [row["train_asr"] for row in rows], marker="o", label="train ASR")
    ax_asr.plot(steps, [row["val_asr"] for row in rows], marker="s", label="val ASR")
    ax_asr.set_title("Attack Success Rate")
    ax_asr.set_xlabel(x_label)
    ax_asr.set_ylabel("ASR")
    ax_asr.set_ylim(0, 1)
    ax_asr.grid(True, alpha=0.25)
    ax_asr.legend()

    fig.tight_layout()
    if save:
        output_dir = Path(config.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_dir / "training_progress.png", dpi=150, bbox_inches="tight")
        fig.savefig(output_dir / "training_progress.svg", bbox_inches="tight")
    if plot_handle is None:
        plot_handle = display(fig, display_id=True)
    else:
        plot_handle.update(fig)
    plt.close(fig)


def live_plot(metrics):
    history.append(metrics.copy())
    render_progress(history)


def refresh_plot_from_history():
    global history
    history = read_history_csv()
    render_progress(history)


history = read_history_csv()
render_progress(history)


In [ ]:
clear_output(wait=True)
_training_output = io.StringIO()
with contextlib.redirect_stdout(_training_output):
    trainer = DePatchTrainer(config)
    history = trainer.fit(callback=live_plot)
